In [1]:
import pandas as pd
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
# Baseline model name (or full path). Prefix with 'baseline:' to pull from
# results/whisper_baseline/whisper_baseline.csv
BASELINE = 'baseline:whisper'

# Comparison model names (without .csv) or full paths — one or more.
COMPARISON_MODELS = [
    'bridge_position_eps_0.5',
    'bridge_dtw_eps_0.5',
    'bridge_dtw_eps_0.5_odesampling',


    # eps, sigma sweep
    'bridge_dtw_fixed_eps_0.3',
    'bridge_dtw_fixed_eps_0.3_ode',
    'bridge_dtw_fixed_eps_0.3_ode_renorm',
    'bridge_dtw_fixed_eps_0.5',
    'bridge_dtw_fixed_eps_0.5_ode',
    'bridge_dtw_fixed_eps_0.5_renorm',
    'bridge_dtw_fixed_eps_0.5_ode_renorm',
    'bridge_dtw_fixed_eps_1.0',
    'bridge_dtw_fixed_eps_1.0_ode',
    'bridge_dtw_fixed_eps_1.0_ode_renorm',

    # x0, sigma sweep
    'bridge_dtw_fixed_x0_0.0_ode',
    'bridge_dtw_fixed_x0_0.0_ode_renorm',
    'bridge_dtw_fixed_x0_0.3',
    'bridge_dtw_fixed_x0_0.3_ode',
    'bridge_dtw_fixed_x0_0.3_ode_renorm',
    'bridge_dtw_fixed_x0_0.5',
    'bridge_dtw_fixed_x0_0.5_ode',
    'bridge_dtw_fixed_x0_0.5_renorm',
    'bridge_dtw_fixed_x0_0.5_ode_renorm',
    'bridge_dtw_fixed_x0_1.0',
    'bridge_dtw_fixed_x0_1.0_ode',
    'bridge_dtw_fixed_x0_1.0_ode_renorm',



    # 'bridge_dtw_fixed_x0_spanish_0.0_ode',
    # 'bridge_dtw_fixed_x0_spanish_0.0_renorm',
    # 'bridge_dtw_fixed_x0_spanish',
    # 'bridge_dtw_fixed_x0_spanish_renorm',
    # 'bridge_dtw_fixed_x0_spanish_ode',
    # 'bridge_dtw_fixed_x0_spanish_ode_renorm',
    
]

# Metric that drives win/draw/loss classification and best/worst ranking.
PRIMARY_METRIC = 'utt_wer'

# Columns to show in comparison tables. Use None to show all shared metric columns.
SHOW_COLS = ['utt_wer']

# Number of rows to show in the best-wins / worst-losses tables.
TOP_N = 20

# Optional filters (set to None to skip) — applied to baseline and all comparison models
FILTER_L1 = None       # e.g. 'Hindi'
FILTER_SPEAKER = None  # e.g. 'THV'

# ── Path resolution ───────────────────────────────────────────────────────────
ROOT = Path("/vol/gpudata/tsv22-fyp/accent-robust-asr")
BRIDGE_DIR   = Path(f'{ROOT}/results/bridge_eval')
STEERING_DIR = Path(f'{ROOT}/results/e2_steering')
BASELINE_DIR = Path(f'{ROOT}/results/whisper_baseline')

def resolve(name: str) -> Path:
    if name.startswith('baseline:'):
        return BASELINE_DIR / 'whisper_baseline.csv'
    p = Path(name)
    if p.suffix == '.csv' and p.exists():
        return p
    for d in [BRIDGE_DIR, STEERING_DIR]:
        candidate = d / f'{name}.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Cannot find CSV for {name!r}')

path_baseline = resolve(BASELINE)
print(f'Baseline: {BASELINE} -> {path_baseline}')
for name in COMPARISON_MODELS:
    print(f'  {name} -> {resolve(name)}')

Baseline: baseline:whisper -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/whisper_baseline/whisper_baseline.csv
  bridge_position_eps_0.5 -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_position_eps_0.5.csv
  bridge_dtw_eps_0.5 -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_eps_0.5.csv
  bridge_dtw_eps_0.5_odesampling -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_eps_0.5_odesampling.csv
  bridge_dtw_fixed_eps_0.3 -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_fixed_eps_0.3.csv
  bridge_dtw_fixed_eps_0.3_ode -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_fixed_eps_0.3_ode.csv
  bridge_dtw_fixed_eps_0.3_ode_renorm -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_fixed_eps_0.3_ode_renorm.csv
  bridge_dtw_fixed_eps_0.5 -> /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/bridge_dtw_fixed_eps_0.5.csv
  bridge_dtw_fi

In [2]:
NON_METRIC = {'utterance_id', 'speaker', 'l1', 'domain', 'wav_path', 'text',
              'prediction', 'reference_norm', 'prediction_norm', 'speaker_type',
              'bridge_split', '_label'}

def load(path: Path, label: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    # normalise column names across different eval scripts
    renames = {'wer': 'utt_wer', 'mer': 'utt_mer', 'per': 'utt_per',
               'whisper_pred': 'prediction', 'whisper_pred_norm': 'prediction_norm'}
    df = df.rename(columns={k: v for k, v in renames.items() if k in df.columns})
    df['_label'] = label
    return df

def apply_filters(df):
    if FILTER_L1 and 'l1' in df.columns:
        df = df[df['l1'] == FILTER_L1]
    if FILTER_SPEAKER and 'speaker' in df.columns:
        df = df[df['speaker'] == FILTER_SPEAKER]
    return df

def compare(da: pd.DataFrame, model_name: str) -> dict:
    """Join filtered baseline rows `da` against `model_name`'s eval CSV and
    precompute win/draw/loss stats, an L1 breakdown, and best/worst tables.

    Columns from the baseline get a `_baseline` suffix, columns from the
    comparison model get a `_bridge` suffix. `{metric}_delta` is always
    `bridge - baseline`: positive means the comparison model is worse than
    the baseline (higher WER), negative means it's better (lower WER).
    """
    db = apply_filters(load(resolve(model_name), model_name))

    metric_cols_a = [c for c in da.columns if c not in NON_METRIC]
    metric_cols_b = [c for c in db.columns if c not in NON_METRIC]
    shared_metrics = [c for c in metric_cols_a if c in metric_cols_b]
    cols_to_show = SHOW_COLS if SHOW_COLS else shared_metrics

    keep_a = ['utterance_id', 'speaker'] + (['l1'] if 'l1' in da.columns else []) + \
             ['text'] + cols_to_show + \
             (['prediction_norm'] if 'prediction_norm' in da.columns else [])
    keep_b = ['utterance_id', 'speaker'] + cols_to_show + \
             (['prediction_norm'] if 'prediction_norm' in db.columns else [])

    merged = da[keep_a].merge(
        db[[c for c in keep_b if c in db.columns]],
        on=['utterance_id', 'speaker'],
        suffixes=('_baseline', '_bridge'),
        how='inner'
    )

    for c in cols_to_show:
        c_base, c_bridge = f'{c}_baseline', f'{c}_bridge'
        if c_base in merged.columns and c_bridge in merged.columns:
            merged[f'{c}_delta'] = merged[c_bridge] - merged[c_base]

    pm_delta = f'{PRIMARY_METRIC}_delta'
    pm_bridge = f'{PRIMARY_METRIC}_bridge'
    is_win  = merged[pm_delta] < 0
    is_loss = merged[pm_delta] > 0
    is_draw = merged[pm_delta] == 0
    is_zero = merged[pm_bridge] == 0
    winloss_counts = {
        'wins':          int(is_win.sum()),
        'corrected':     int((is_win & is_zero).sum()),
        'improved':      int((is_win & ~is_zero).sum()),
        'draws':         int(is_draw.sum()),
        'draws_zero':    int((is_draw & is_zero).sum()),
        'draws_nonzero': int((is_draw & ~is_zero).sum()),
        'losses':        int(is_loss.sum()),
    }

    l1_table = None
    if 'l1' in merged.columns:
        rows = []
        for l1, g in merged.groupby('l1'):
            row = {'l1': l1, 'n': len(g)}
            for c in cols_to_show:
                c_base, c_bridge = f'{c}_baseline', f'{c}_bridge'
                row[f'{c}_baseline'] = g[c_base].mean() * 100
                row[f'{c}_bridge']   = g[c_bridge].mean() * 100
                row[f'{c}_delta']    = (g[c_bridge].mean() - g[c_base].mean()) * 100
            rows.append(row)
        l1_table = pd.DataFrame(rows).set_index('l1').round(2)

    text_cols   = ['utterance_id', 'speaker'] + (['l1'] if 'l1' in merged.columns else []) + ['text']
    pred_cols   = [c for c in merged.columns if 'prediction_norm' in c]
    metric_disp = [c for c in merged.columns if any(c.startswith(m) for m in cols_to_show)]
    browse_cols = text_cols + pred_cols + metric_disp

    best_wins    = merged.sort_values(pm_delta, ascending=True).head(TOP_N)[browse_cols].reset_index(drop=True)
    worst_losses = merged.sort_values(pm_delta, ascending=False).head(TOP_N)[browse_cols].reset_index(drop=True)
    for col in metric_disp:
        best_wins[col]    = (best_wins[col]    * 100).round(2)
        worst_losses[col] = (worst_losses[col] * 100).round(2)

    return {
        'name': model_name,
        'merged': merged,
        'n': len(merged),
        'cols_to_show': cols_to_show,
        'winloss_counts': winloss_counts,
        'l1_table': l1_table,
        'best_wins': best_wins,
        'worst_losses': worst_losses,
    }

da_baseline = apply_filters(load(path_baseline, BASELINE))
print(f'Baseline {BASELINE!r}: {len(da_baseline)} rows')

results = [compare(da_baseline, name) for name in COMPARISON_MODELS]

print(f'\nComputed {len(results)} comparison(s) against baseline:')
for r in results:
    wl = r['winloss_counts']
    print(f"  {r['name']:35s} n={r['n']:5d}  wins={wl['wins']:4d} (corrected={wl['corrected']:4d}, improved={wl['improved']:4d})"
          f"  draws={wl['draws']:4d} (zero={wl['draws_zero']:4d}, nonzero={wl['draws_nonzero']:4d})  losses={wl['losses']:4d}")

Baseline 'baseline:whisper': 31395 rows

Computed 25 comparison(s) against baseline:
  bridge_position_eps_0.5             n= 7796  wins= 152 (corrected=  44, improved= 108)  draws=1183 (zero= 747, nonzero= 436)  losses=6460
  bridge_dtw_eps_0.5                  n= 7796  wins= 635 (corrected= 180, improved= 455)  draws=5935 (zero=3156, nonzero=2779)  losses=1225
  bridge_dtw_eps_0.5_odesampling      n= 7796  wins= 607 (corrected= 168, improved= 439)  draws=6055 (zero=3188, nonzero=2867)  losses=1133
  bridge_dtw_fixed_eps_0.3            n= 7796  wins= 899 (corrected= 269, improved= 630)  draws=6033 (zero=3235, nonzero=2798)  losses= 863
  bridge_dtw_fixed_eps_0.3_ode        n= 7796  wins= 912 (corrected= 269, improved= 643)  draws=6126 (zero=3276, nonzero=2850)  losses= 757
  bridge_dtw_fixed_eps_0.3_ode_renorm n= 7796  wins= 899 (corrected= 256, improved= 643)  draws=6145 (zero=3273, nonzero=2872)  losses= 751
  bridge_dtw_fixed_eps_0.5            n= 7796  wins= 967 (corrected= 303, i

In [4]:
from IPython.display import display, Markdown

pm = PRIMARY_METRIC
raw_rows, pct_rows = [], []
for r in results:
    merged, wl, n = r['merged'], r['winloss_counts'], r['n']
    baseline_mean = merged[f'{pm}_baseline'].mean()
    bridge_mean   = merged[f'{pm}_bridge'].mean()

    raw_rows.append({
        'model': r['name'],
        'n': n,
        f'{pm}_baseline_%': baseline_mean * 100,
        f'{pm}_bridge_%':   bridge_mean   * 100,
        f'{pm}_delta_%':    (bridge_mean - baseline_mean) * 100,
        'wins': wl['wins'],
        'corrected': wl['corrected'],
        'improved': wl['improved'],
        'draws': wl['draws'],
        'draws_zero': wl['draws_zero'],
        'draws_nonzero': wl['draws_nonzero'],
        'losses': wl['losses'],
    })
    pct_rows.append({
        'model': r['name'],
        'win_%':           wl['wins']          / n * 100,
        'corrected_%':     wl['corrected']     / n * 100,
        'improved_%':      wl['improved']      / n * 100,
        'draw_%':          wl['draws']         / n * 100,
        'draws_zero_%':    wl['draws_zero']    / n * 100,
        'draws_nonzero_%': wl['draws_nonzero'] / n * 100,
        'loss_%':          wl['losses']        / n * 100,
    })

overview_raw = pd.DataFrame(raw_rows).set_index('model').round(2)
overview_pct = pd.DataFrame(pct_rows).set_index('model').round(2)

display(Markdown('**Raw counts (WER in %; wins split into corrected [→0 WER] / improved [still nonzero]; '
                 'draws split into zero / nonzero):**'))
display(overview_raw)
display(Markdown('**Percentages (of compared utterances `n`):**'))
display(overview_pct)

**Raw counts (WER in %; wins split into corrected [→0 WER] / improved [still nonzero]; draws split into zero / nonzero):**

,n,utt_wer_baseline_%,utt_wer_bridge_%,utt_wer_delta_%,wins,corrected,improved,draws,draws_zero,draws_nonzero,losses
model,,,,,,,,,,,
bridge_position_eps_0.5,7796,15.4,224.29,208.89,152,44,108,1183,747,436,6460
bridge_dtw_eps_0.5,7796,15.4,19.15,3.74,635,180,455,5935,3156,2779,1225
bridge_dtw_eps_0.5_odesampling,7796,15.4,18.89,3.49,607,168,439,6055,3188,2867,1133
bridge_dtw_fixed_eps_0.3,7796,15.4,16.56,1.16,899,269,630,6033,3235,2798,863
bridge_dtw_fixed_eps_0.3_ode,7796,15.4,15.30,-0.10,912,269,643,6126,3276,2850,757
bridge_dtw_fixed_eps_0.3_ode_renorm,7796,15.4,14.96,-0.44,899,256,643,6145,3273,2872,751
bridge_dtw_fixed_eps_0.5,7796,15.4,22.31,6.90,967,303,664,5804,3194,2610,1024
bridge_dtw_fixed_eps_0.5_ode,7796,15.4,18.54,3.14,973,290,683,6001,3249,2752,821
bridge_dtw_fixed_eps_0.5_renorm,7796,15.4,20.61,5.20,994,309,685,5787,3187,2600,1014


**Percentages (of compared utterances `n`):**

,win_%,corrected_%,improved_%,draw_%,draws_zero_%,draws_nonzero_%,loss_%
model,,,,,,,
bridge_position_eps_0.5,1.95,0.56,1.39,15.17,9.58,5.59,82.86
bridge_dtw_eps_0.5,8.15,2.31,5.84,76.13,40.48,35.65,15.71
bridge_dtw_eps_0.5_odesampling,7.79,2.15,5.63,77.67,40.89,36.78,14.53
bridge_dtw_fixed_eps_0.3,11.53,3.45,8.08,77.39,41.50,35.89,11.07
bridge_dtw_fixed_eps_0.3_ode,11.70,3.45,8.25,78.58,42.02,36.56,9.71
bridge_dtw_fixed_eps_0.3_ode_renorm,11.53,3.28,8.25,78.82,41.98,36.84,9.63
bridge_dtw_fixed_eps_0.5,12.40,3.89,8.52,74.45,40.97,33.48,13.13
bridge_dtw_fixed_eps_0.5_ode,12.48,3.72,8.76,76.98,41.68,35.30,10.53
bridge_dtw_fixed_eps_0.5_renorm,12.75,3.96,8.79,74.23,40.88,33.35,13.01


In [5]:
from IPython.display import display, Markdown

pm_delta = f'{PRIMARY_METRIC}_delta'
pm_bridge = f'{PRIMARY_METRIC}_bridge'

l1_pct_rows = []
for r in results:
    merged = r['merged']
    if 'l1' not in merged.columns:
        continue
    for l1, g in merged.groupby('l1'):
        n = len(g)
        l1_pct_rows.append({
            'model':  r['name'],
            'l1':     l1,
            'win_%':  (g[pm_delta] < 0).sum() / n * 100,
            'draw_%': (g[pm_delta] == 0).sum() / n * 100,
            'loss_%': (g[pm_delta] > 0).sum() / n * 100,
        })

l1_pct_df = pd.DataFrame(l1_pct_rows)

for col, label in [('win_%', 'Win %'), ('loss_%', 'Loss %'), ('draw_%', 'Draw %')]:
    pivot = l1_pct_df.pivot(index='model', columns='l1', values=col).round(2)
    display(Markdown(f'**{label} by L1:**'))
    display(pivot)

**Win % by L1:**

l1,Arabic,Chinese,English,Hindi,Korean,Spanish,Vietnamese
model,,,,,,,
bridge_dtw_eps_0.5,6.71,10.97,2.21,6.36,5.57,12.31,13.34
bridge_dtw_eps_0.5_odesampling,5.74,11.33,1.77,5.65,5.84,11.72,12.90
bridge_dtw_fixed_eps_0.3,9.28,18.67,2.39,7.16,7.78,17.68,18.46
bridge_dtw_fixed_eps_0.3_ode,9.36,17.26,2.39,7.42,8.22,18.97,19.08
bridge_dtw_fixed_eps_0.3_ode_renorm,9.54,17.17,2.12,6.98,7.60,19.46,18.73
bridge_dtw_fixed_eps_0.5,10.60,18.85,2.65,7.95,8.49,18.97,20.05
bridge_dtw_fixed_eps_0.5_ode,10.42,19.03,2.39,7.86,9.20,19.07,20.14
bridge_dtw_fixed_eps_0.5_ode_renorm,11.04,19.38,2.30,7.69,8.49,20.36,21.11
bridge_dtw_fixed_eps_0.5_renorm,11.22,19.03,3.00,7.95,8.75,19.27,20.76


**Loss % by L1:**

l1,Arabic,Chinese,English,Hindi,Korean,Spanish,Vietnamese
model,,,,,,,
bridge_dtw_eps_0.5,13.16,25.84,4.59,7.07,9.02,24.53,26.77
bridge_dtw_eps_0.5_odesampling,10.25,25.66,3.80,6.36,8.49,23.34,24.82
bridge_dtw_fixed_eps_0.3,9.54,12.48,2.74,5.48,7.69,16.78,23.41
bridge_dtw_fixed_eps_0.3_ode,8.39,11.86,1.86,3.89,6.98,13.90,21.55
bridge_dtw_fixed_eps_0.3_ode_renorm,8.57,11.68,1.86,4.06,6.37,15.49,20.05
bridge_dtw_fixed_eps_0.5,11.66,16.81,3.27,6.71,9.73,19.27,25.18
bridge_dtw_fixed_eps_0.5_ode,9.10,12.92,1.86,4.33,7.60,16.68,21.91
bridge_dtw_fixed_eps_0.5_ode_renorm,8.92,11.59,1.77,3.98,6.98,15.69,20.23
bridge_dtw_fixed_eps_0.5_renorm,11.40,15.93,3.09,6.80,10.34,18.87,25.27


**Draw % by L1:**

l1,Arabic,Chinese,English,Hindi,Korean,Spanish,Vietnamese
model,,,,,,,
bridge_dtw_eps_0.5,80.12,63.19,93.11,86.57,85.41,63.16,59.89
bridge_dtw_eps_0.5_odesampling,84.01,63.01,94.35,87.99,85.68,64.95,62.28
bridge_dtw_fixed_eps_0.3,81.18,68.85,94.79,87.37,84.53,65.54,58.13
bridge_dtw_fixed_eps_0.3_ode,82.24,70.88,95.67,88.69,84.79,67.13,59.36
bridge_dtw_fixed_eps_0.3_ode_renorm,81.89,71.15,95.94,88.96,86.03,65.04,61.22
bridge_dtw_fixed_eps_0.5,77.74,64.34,93.99,85.34,81.79,61.77,54.77
bridge_dtw_fixed_eps_0.5_ode,80.48,68.05,95.67,87.81,83.20,64.25,57.95
bridge_dtw_fixed_eps_0.5_ode_renorm,80.04,69.03,95.85,88.34,84.53,63.95,58.66
bridge_dtw_fixed_eps_0.5_renorm,77.39,65.04,93.82,85.25,80.90,61.87,53.98


In [17]:
import jiwer
from IPython.display import display, Markdown

def _cwer(refs, preds):
    return float(jiwer.wer([str(r) for r in refs], [str(p) for p in preds]))

def _cmer(refs, preds):
    return float(jiwer.mer([str(r) for r in refs], [str(p) for p in preds]))

corpus_rows, l1_rows = [], []
for r in results:
    bridge_df = pd.read_csv(resolve(r['name'])).rename(columns={'wer': 'utt_wer'})
    m = r['merged'][['utterance_id', 'speaker', 'l1']].merge(
        bridge_df[['utterance_id', 'speaker', 'reference_norm', 'prediction_norm']],
        on=['utterance_id', 'speaker'], how='inner')
    m = m.merge(
        da_baseline[['utterance_id', 'speaker', 'prediction_norm']].rename(
            columns={'prediction_norm': 'pred_base'}),
        on=['utterance_id', 'speaker'], how='inner')

    refs         = m['reference_norm'].fillna('').tolist()
    preds_base   = m['pred_base'].fillna('').tolist()
    preds_bridge = m['prediction_norm'].fillna('').tolist()

    corpus_rows.append({
        'model':            r['name'],
        'wer_baseline_%':   _cwer(refs, preds_base)   * 100,
        'wer_bridge_%':     _cwer(refs, preds_bridge) * 100,
        'wer_delta_%':      (_cwer(refs, preds_bridge) - _cwer(refs, preds_base)) * 100,
        'mer_baseline_%':   _cmer(refs, preds_base)   * 100,
        'mer_bridge_%':     _cmer(refs, preds_bridge) * 100,
        'mer_delta_%':      (_cmer(refs, preds_bridge) - _cmer(refs, preds_base)) * 100,
    })

    for l1, g in m.groupby('l1'):
        refs_l1   = g['reference_norm'].fillna('').tolist()
        base_l1   = g['pred_base'].fillna('').tolist()
        bridge_l1 = g['prediction_norm'].fillna('').tolist()
        l1_rows.append({
            'model':          r['name'],
            'l1':             l1,
            'wer_baseline_%': _cwer(refs_l1, base_l1)   * 100,
            'wer_bridge_%':   _cwer(refs_l1, bridge_l1) * 100,
            'wer_delta_%':    (_cwer(refs_l1, bridge_l1) - _cwer(refs_l1, base_l1)) * 100,
            'mer_baseline_%': _cmer(refs_l1, base_l1)   * 100,
            'mer_bridge_%':   _cmer(refs_l1, bridge_l1) * 100,
            'mer_delta_%':    (_cmer(refs_l1, bridge_l1) - _cmer(refs_l1, base_l1)) * 100,
        })

display(Markdown('### Corpus WER + MER % (jiwer — total errors / total reference words)'))
display(pd.DataFrame(corpus_rows).set_index('model').round(2))

l1_df = pd.DataFrame(l1_rows)
baseline_wer = (l1_df[l1_df['model'] == l1_df['model'].iloc[0]]
                .set_index('l1')['wer_baseline_%'].rename('baseline'))
baseline_mer = (l1_df[l1_df['model'] == l1_df['model'].iloc[0]]
                .set_index('l1')['mer_baseline_%'].rename('baseline'))

for metric, bl in [('wer', baseline_wer)]: #, ('mer', baseline_mer)]:
    display(Markdown(f'### Per-L1 corpus {metric.upper()} % (bridge; baseline row for reference)'))
    pivot = l1_df.pivot(index='model', columns='l1', values=f'{metric}_bridge_%').round(2)
    display(pd.concat([bl.to_frame().T.rename(index={'baseline': 'baseline'}), pivot]))

    display(Markdown(f'### Per-L1 corpus {metric.upper()} % delta (bridge − baseline; negative = improvement)'))
    display(l1_df.pivot(index='model', columns='l1', values=f'{metric}_delta_%').round(2))

### Corpus WER + MER % (jiwer — total errors / total reference words)

,wer_baseline_%,wer_bridge_%,wer_delta_%,mer_baseline_%,mer_bridge_%,mer_delta_%
model,,,,,,
bridge_dtw_eps_0.5,14.71,19.10,4.39,14.5,18.34,3.84
bridge_dtw_eps_0.5_odesampling,14.71,18.85,4.14,14.5,18.11,3.60
bridge_dtw_fixed_eps_0.3,14.71,15.60,0.89,14.5,15.24,0.74
bridge_dtw_fixed_eps_0.3_ode,14.71,14.85,0.14,14.5,14.59,0.09
bridge_dtw_fixed_eps_0.3_ode_renorm,14.71,14.40,-0.31,14.5,14.22,-0.29
bridge_dtw_fixed_eps_0.5,14.71,21.86,7.15,14.5,20.18,5.67
bridge_dtw_fixed_eps_0.5_ode,14.71,17.73,3.02,14.5,16.94,2.44
bridge_dtw_fixed_eps_0.5_renorm,14.71,20.15,5.44,14.5,18.88,4.37
bridge_dtw_fixed_eps_0.5_ode_renorm,14.71,14.17,-0.54,14.5,14.00,-0.50


### Per-L1 corpus WER % (bridge; baseline row for reference)

l1,Arabic,Chinese,English,Hindi,Korean,Spanish,Vietnamese
baseline,12.645392,19.19625,3.815341,6.55265,7.901284,22.647421,31.020084
bridge_dtw_eps_0.5,13.590000,30.86000,4.170000,6.83000,8.530000,28.800000,41.950000
bridge_dtw_eps_0.5_odesampling,13.190000,30.54000,4.130000,6.70000,8.320000,28.480000,41.570000
bridge_dtw_fixed_eps_0.3,12.560000,18.24000,3.770000,6.21000,7.940000,22.680000,38.490000
bridge_dtw_fixed_eps_0.3_ode,12.430000,18.25000,3.690000,6.02000,7.770000,25.460000,31.430000
bridge_dtw_fixed_eps_0.3_ode_renorm,12.280000,18.22000,3.730000,6.09000,7.760000,22.230000,31.270000
bridge_dtw_fixed_eps_0.5,16.940000,19.01000,3.850000,6.30000,8.200000,46.960000,54.430000
bridge_dtw_fixed_eps_0.5_ode,12.270000,22.37000,3.630000,6.07000,7.720000,31.220000,42.230000
bridge_dtw_fixed_eps_0.5_ode_renorm,12.020000,17.79000,3.640000,5.99000,7.680000,22.190000,30.730000
bridge_dtw_fixed_eps_0.5_renorm,16.830000,18.61000,3.760000,6.31000,8.250000,27.920000,60.080000


### Per-L1 corpus WER % delta (bridge − baseline; negative = improvement)

l1,Arabic,Chinese,English,Hindi,Korean,Spanish,Vietnamese
model,,,,,,,
bridge_dtw_eps_0.5,0.94,11.67,0.35,0.28,0.63,6.15,10.93
bridge_dtw_eps_0.5_odesampling,0.55,11.35,0.31,0.15,0.42,5.83,10.55
bridge_dtw_fixed_eps_0.3,-0.09,-0.96,-0.05,-0.34,0.04,0.03,7.47
bridge_dtw_fixed_eps_0.3_ode,-0.22,-0.95,-0.13,-0.54,-0.13,2.81,0.41
bridge_dtw_fixed_eps_0.3_ode_renorm,-0.37,-0.98,-0.09,-0.47,-0.14,-0.41,0.25
bridge_dtw_fixed_eps_0.5,4.29,-0.19,0.03,-0.25,0.30,24.31,23.41
bridge_dtw_fixed_eps_0.5_ode,-0.38,3.17,-0.19,-0.49,-0.18,8.57,11.21
bridge_dtw_fixed_eps_0.5_ode_renorm,-0.63,-1.41,-0.18,-0.57,-0.22,-0.46,-0.29
bridge_dtw_fixed_eps_0.5_renorm,4.19,-0.59,-0.06,-0.24,0.35,5.27,29.06


In [18]:
import numpy as np
from IPython.display import display, Markdown

# ── Bootstrap 95% CI on WER delta ────────────────────────────────────────────
# Paired bootstrap: resample utterance pairs (baseline_i, bridge_i) together,
# preserving per-utterance correlation. Reports mean delta and 95% CI on
# mean(utt_wer_bridge) − mean(utt_wer_baseline), in percentage points.

N_BOOTSTRAP = 10_000
_RNG = np.random.default_rng(42)

def bootstrap_wer_delta_ci(utt_wer_base, utt_wer_bridge, n_boot=N_BOOTSTRAP, alpha=0.05):
    """Paired bootstrap CI on mean WER delta (bridge − baseline), in %."""
    base   = np.asarray(utt_wer_base,   dtype=float) * 100
    bridge = np.asarray(utt_wer_bridge, dtype=float) * 100
    n      = len(base)
    idx    = _RNG.integers(0, n, size=(n_boot, n))
    deltas = bridge[idx].mean(axis=1) - base[idx].mean(axis=1)
    lo, hi = np.percentile(deltas, [alpha / 2 * 100, (1 - alpha / 2) * 100])
    return float(deltas.mean()), float(lo), float(hi)

def _sig(lo, hi):
    if hi < 0: return "✓ sig. improvement"
    if lo > 0: return "✗ sig. degradation"
    return "— inconclusive"

rows = []
for r in results:
    merged = r['merged']
    base_col   = 'utt_wer_baseline'
    bridge_col = 'utt_wer_bridge'
    if base_col not in merged.columns or bridge_col not in merged.columns:
        continue

    valid = merged[[base_col, bridge_col]].dropna()
    mean_d, lo, hi = bootstrap_wer_delta_ci(valid[base_col], valid[bridge_col])
    rows.append({
        'model':        r['name'],
        'n':            len(valid),
        'mean_delta_%': round(mean_d, 3),
        'ci_lo_%':      round(lo, 3),
        'ci_hi_%':      round(hi, 3),
        'verdict':      _sig(lo, hi),
    })

boot_df = pd.DataFrame(rows).set_index('model')

display(Markdown(
    "### Bootstrap 95% CI — WER delta (bridge − baseline, %WER, N=10,000 resamples)\n"
    "Uses mean utterance-level WER (equal weight per utterance).\n"
    "`✓ sig. improvement` = CI entirely < 0 &nbsp;&nbsp;"
    "`✗ sig. degradation` = CI entirely > 0 &nbsp;&nbsp;"
    "`— inconclusive` = CI straddles 0"
))
display(boot_df)


### Bootstrap 95% CI — WER delta (bridge − baseline, %WER, N=10,000 resamples)
Uses mean utterance-level WER (equal weight per utterance).
`✓ sig. improvement` = CI entirely < 0 &nbsp;&nbsp;`✗ sig. degradation` = CI entirely > 0 &nbsp;&nbsp;`— inconclusive` = CI straddles 0

,n,mean_delta_%,ci_lo_%,ci_hi_%,verdict
model,,,,,
bridge_dtw_eps_0.5,7795,3.729,2.013,5.868,✗ sig. degradation
bridge_dtw_eps_0.5_odesampling,7795,3.485,1.739,5.691,✗ sig. degradation
bridge_dtw_fixed_eps_0.3,7795,1.157,-0.268,3.251,— inconclusive
bridge_dtw_fixed_eps_0.3_ode,7795,-0.098,-0.576,0.660,— inconclusive
bridge_dtw_fixed_eps_0.3_ode_renorm,7795,-0.442,-0.650,-0.233,✓ sig. improvement
bridge_dtw_fixed_eps_0.5,7795,6.885,3.337,10.960,✗ sig. degradation
bridge_dtw_fixed_eps_0.5_ode,7795,3.129,0.480,6.470,✗ sig. degradation
bridge_dtw_fixed_eps_0.5_renorm,7795,5.206,2.089,8.978,✗ sig. degradation
bridge_dtw_fixed_eps_0.5_ode_renorm,7795,-0.700,-0.906,-0.489,✓ sig. improvement


In [19]:
# ── Save tables ───────────────────────────────────────────────────────────────
# Subfolder under results/eval_tables/ — e.g. "all", "spanish", "hindi"
SAVE_SUBFOLDER = "all"

save_dir = ROOT / "results" / "bridge_eval" / "summary_tables" / SAVE_SUBFOLDER
save_dir.mkdir(parents=True, exist_ok=True)

# Win/draw/loss summary (overall)
overview_raw.to_csv(save_dir / "summary_raw.csv")
overview_pct.to_csv(save_dir / "summary_pct.csv")

# Win/draw/loss by L1
for col, name in [('win_%', 'win'), ('loss_%', 'loss'), ('draw_%', 'draw')]:
    l1_pct_df.pivot(index='model', columns='l1', values=col).round(2).to_csv(
        save_dir / f"summary_pct_by_l1_{name}.csv")

# Corpus WER + MER
corpus_df = pd.DataFrame(corpus_rows).set_index('model').round(2)
corpus_df.to_csv(save_dir / "corpus_wer_mer.csv")

# Per-L1 breakdowns
for metric in ['wer', 'mer']:
    l1_df.pivot(index='model', columns='l1', values=f'{metric}_bridge_%').round(2).to_csv(
        save_dir / f"per_l1_{metric}_pct.csv")
    l1_df.pivot(index='model', columns='l1', values=f'{metric}_delta_%').round(2).to_csv(
        save_dir / f"per_l1_{metric}_delta.csv")

# Bootstrap CI
boot_df.to_csv(save_dir / "bootstrap_ci_wer.csv")

print(f"Saved to {save_dir}/")
for f in sorted(save_dir.glob("*.csv")):
    print(f"  {f.name}")


Saved to /vol/gpudata/tsv22-fyp/accent-robust-asr/results/bridge_eval/summary_tables/all/
  bootstrap_ci_wer.csv
  corpus_wer_mer.csv
  per_l1_mer_delta.csv
  per_l1_mer_pct.csv
  per_l1_wer_delta.csv
  per_l1_wer_pct.csv
  summary_pct.csv
  summary_pct_by_l1_draw.csv
  summary_pct_by_l1_loss.csv
  summary_pct_by_l1_win.csv
  summary_raw.csv


In [12]:
from IPython.display import display, Markdown

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 200)

# Set to None to browse all models, or list specific names to restrict.
BROWSE_MODELS = [
    'bridge_dtw_fixed_eps_0.5_ode_renorm',
    'bridge_dtw_fixed_eps_0.3_ode_renorm',
]

browse_results = [r for r in results if BROWSE_MODELS is None or r['name'] in BROWSE_MODELS]

for r in browse_results:
    display(Markdown(f"## {r['name']}"))

    wl, n = r['winloss_counts'], r['n']
    print(f"Utterances compared: {n}")
    print(f"  wins   (model better than baseline): {wl['wins']:5d}  ({wl['wins']/n*100:5.2f}%)")
    print(f"  draws  (tied):                       {wl['draws']:5d}  ({wl['draws']/n*100:5.2f}%)")
    print(f"  losses (model worse than baseline):  {wl['losses']:5d}  ({wl['losses']/n*100:5.2f}%)")

    if r['l1_table'] is not None:
        display(Markdown("**By L1:**"))
        display(r['l1_table'])

    display(Markdown(f"**Best wins — top {TOP_N} by `{PRIMARY_METRIC}` delta (model beats baseline most):**"))
    display(r['best_wins'])

    display(Markdown(f"**Worst losses — top {TOP_N} by `{PRIMARY_METRIC}` delta (model loses to baseline most):**"))
    display(r['worst_losses'])


## bridge_dtw_fixed_eps_0.3_ode_renorm

Utterances compared: 7796
  wins   (model better than baseline):   899  (11.53%)
  draws  (tied):                        6145  (78.82%)
  losses (model worse than baseline):    751  ( 9.63%)


**By L1:**

,n,utt_wer_baseline,utt_wer_bridge,utt_wer_delta
l1,,,,
Arabic,1132,13.08,12.69,-0.39
Chinese,1130,20.10,18.84,-1.27
English,1132,4.02,3.91,-0.11
Hindi,1132,7.08,6.47,-0.61
Korean,1131,8.55,8.27,-0.28
Spanish,1007,23.64,22.96,-0.68
Vietnamese,1132,32.23,32.45,0.22


**Best wins — top 20 by `utt_wer` delta (model beats baseline most):**

...

,n,utt_wer_baseline,utt_wer_bridge,utt_wer_delta
l1,,,,
Arabic,1132,13.08,12.36,-0.72
Chinese,1130,20.10,18.31,-1.79
English,1132,4.02,3.82,-0.20
Hindi,1132,7.08,6.38,-0.70
Korean,1131,8.55,8.15,-0.41
Spanish,1007,23.64,23.02,-0.63
Vietnamese,1132,32.23,31.78,-0.45


**Best wins — top 20 by `utt_wer` delta (model beats baseline most):**

,utterance_id,speaker,l1,text,prediction_norm_baseline,prediction_norm_bridge,utt_wer_baseline,utt_wer_bridge,utt_wer_delta
0,arctic_a0381,SVBI,Hindi,My name's Ferguson,my name is fargusson,my names ferguson,100.00,0.00,-100.00
1,arctic_b0425,BDL,English,"There were orange-green, gold-green, and a copper-green.",there were orange green gold green and a copper green,there were orangegreen goldgreen and a coppergreen,85.71,0.00,-85.71
2,arctic_a0155,HQTV,Vietnamese,Won't you draw up gentlemen,one youve rarred and the other youve chandlemen,won you rob and chanderman,160.00,80.00,-80.00
3,arctic_b0319,HQTV,Vietnamese,Daylight was tired profoundly tired,they lied to a tire a foully tire,they lied were tired refowlly tired,160.00,80.00,-80.00
4,arctic_a0589,BWC,Chinese,I was sick once typhoid,i will seek once time for it,i was sick once typhoic,100.00,20.00,-80.00
5,arctic_b0536,BWC,Chinese,Typhoid did I tell you,what type of fight did i tell you,typhoid did i tell you,80.00,0.00,-80.00
6,arctic_b0207,ZHAA,Arabic,I'm as good as a man she urged,amasgut ezaman she urged,im as good as a man she urged,75.00,0.00,-75.00
7,arctic_b0299,HQTV,Vietnamese,Miss Brodie's smile was slightly sarcastic,misbroadies my words like this sarcastic,miss brodys smile was slightly sarcastic,83.33,16.67,-66.67
8,arctic_b0257,HQTV,Vietnamese,Tudor surveyed him with withering disgust,to the survey he was with three discussed,to the survey him with withering disgust,116.67,50.00,-66.67
9,arctic_a0119,HQTV,Vietnamese,Jeanne was turning the bow shoreward,genie would turn in the bull straw world,ginny was turning the bow straw war,116.67,50.00,-66.67


**Worst losses — top 20 by `utt_wer` delta (model loses to baseline most):**

,utterance_id,speaker,l1,text,prediction_norm_baseline,prediction_norm_bridge,utt_wer_baseline,utt_wer_bridge,utt_wer_delta
0,arctic_b0449,HQTV,Vietnamese,Also she wouldn't walk,also she wouldnt walk,also she would the world,0.00,75.00,75.00
1,arctic_a0028,HQTV,Vietnamese,Robbery bribery fraud,rubbery bribery fraud,rubbery prypory froat,33.33,100.00,66.67
2,arctic_b0268,SVBI,Hindi,Saxon nodded and the boy frowned,saxon nordic and the boy frowned,sacks are knotted and the boyfriend,16.67,83.33,66.67
3,arctic_a0557,HQTV,Vietnamese,The last refugee had passed,the last refutious had passed,the last refuture is head pass,20.00,80.00,60.00
4,arctic_a0187,HQTV,Vietnamese,Ahead of them they saw a glimmer of sunshine,ahead of them i saw clean machine size,i had to tell my soul clean my sons eyes,55.56,111.11,55.56
5,arctic_a0407,HQTV,Vietnamese,Mercedes screamed cried laughed and manifested the chaotic abandonment of hy...,mercedes screamed cried laughed and manifested the chaotic abundance of hyst...,messed as scream cry laugh and manifested the childlike abundant of hysteria,9.09,63.64,54.55
6,arctic_a0329,HQTV,Vietnamese,Ah indeed,are indeed,all right indeed,50.00,100.00,50.00
7,arctic_a0050,BWC,Chinese,In spite of their absurdity the words affected Philip curiously,in spirit of their assertedity the words affect philip curiously,its be off there as third duty the words affect philip curiously,30.00,80.00,50.00
8,arctic_a0419,BWC,Chinese,The Portuguese boy passed the Hawaiian,the portuguese boy passed the hawaii,the pachawgis fly past the hawaii,16.67,66.67,50.00
9,arctic_b0180,HQTV,Vietnamese,I I beg pardon he drawled,hi i beg pardon evrol,hi im bec parlin evrol,50.00,100.00,50.00


In [7]:
# ── Look up specific utterances across all models ─────────────────────────────
# Add (utterance_id, speaker) pairs here to see how the baseline and every
# comparison model handled them, side by side.
LOOKUP_UTTERANCES = [
    ('arctic_a0484', 'BDL'),
    ('arctic_b0319', 'HQTV'),
]

lookup_keys = pd.DataFrame(LOOKUP_UTTERANCES, columns=['utterance_id', 'speaker'])

base_cols = ['utterance_id', 'speaker'] + (['l1'] if 'l1' in da_baseline.columns else []) + \
            ['text', 'prediction_norm', PRIMARY_METRIC]
wide = lookup_keys.merge(da_baseline[base_cols], on=['utterance_id', 'speaker'], how='left')
wide = wide.rename(columns={
    'prediction_norm': f'prediction_norm_{BASELINE}',
    PRIMARY_METRIC: f'{PRIMARY_METRIC}_{BASELINE}',
})

for r in results:
    sub = r['merged'][['utterance_id', 'speaker', 'prediction_norm_bridge', f'{PRIMARY_METRIC}_bridge']].rename(columns={
        'prediction_norm_bridge': f'prediction_norm_{r["name"]}',
        f'{PRIMARY_METRIC}_bridge': f'{PRIMARY_METRIC}_{r["name"]}',
    })
    wide = wide.merge(sub, on=['utterance_id', 'speaker'], how='left')

wide

,utterance_id,speaker,l1,text,prediction_norm_baseline:whisper,utt_wer_baseline:whisper,prediction_norm_bridge_dtw_eps_0.5,utt_wer_bridge_dtw_eps_0.5,prediction_norm_bridge_dtw_eps_0.5_odesampling,utt_wer_bridge_dtw_eps_0.5_odesampling,...,prediction_norm_bridge_dtw_fixed_x0_0.5_ode,utt_wer_bridge_dtw_fixed_x0_0.5_ode,prediction_norm_bridge_dtw_fixed_x0_0.5_ode_renorm,utt_wer_bridge_dtw_fixed_x0_0.5_ode_renorm,prediction_norm_bridge_dtw_fixed_x0_1.0,utt_wer_bridge_dtw_fixed_x0_1.0,prediction_norm_bridge_dtw_fixed_x0_1.0_ode,utt_wer_bridge_dtw_fixed_x0_1.0_ode,prediction_norm_bridge_dtw_fixed_x0_1.0_ode_renorm,utt_wer_bridge_dtw_fixed_x0_1.0_ode_renorm
0,arctic_a0484,BDL,English,No-sir-ee.,no surrey,2.0,no sirree,2.0,no sirree,2.0,...,no surrey,2.0,no surrey,2.0,no sirree,2.0,no surrey,2.0,no surrey,2.0
1,arctic_b0319,HQTV,Vietnamese,Daylight was tired profoundly tired,they lied to a tire a foully tire,1.6,they lied were tired profoundly tired,0.6,they lied were tired were profoundly tired,0.8,...,they lied were tied refowlied tied,1.2,they lied were tied refowlly tied,1.2,they light with tide a firefly,1.2,they light with tide refowlly tide,1.2,they light with tide refowlly tide,1.2
